# Tolkien Anchored Sentiment — Held-out Evaluation (Precision / Recall / F1)

This notebook evaluates an anchor-based sentiment scoring procedure on a held-out set of manually labelled sentences.

**Core idea**
1. Build a **positive anchor centroid** and **negative anchor centroid** from *anchor* sentences.
2. For each **test** sentence, embed it and compare cosine similarity to both centroids.
3. Predict `positive` if it's closer to the positive centroid, else `negative`.
4. Compute metrics: **confusion matrix, accuracy, precision, recall, F1**.

> Tip: Keep anchors and test sentences **disjoint** to avoid leakage.


In [ ]:
# Cell — Imports
import os
import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
from pathlib import Path

# If you use SentenceTransformers:
from sentence_transformers import SentenceTransformer


> In this notebook, the term “anchor sentence” refers to a short reference example used in the anchored sentiment evaluation procedure. It should not be confused with the semantic-axis projection method used in the separate semantic-axis notebook.

## Input data format

This notebook evaluates whether different sentence-embedding models can distinguish between positive and negative sentiment examples drawn from *The Lord of the Rings*.

The manually selected sentences are divided into two roles:

1. **Anchor sentences**  
   These are short positive and negative reference sentences used by the sentiment-scoring procedure. They are not treated as test examples, because they contribute directly to the scoring setup.

2. **Gold-labelled test sentences**  
   These are held-out positive and negative examples used only for evaluation. They are not used as anchors. This separation avoids evaluating the models on sentences that were already used as reference examples.

In other words, the anchor sentences provide the positive and negative reference points for the method, while the test sentences provide an independent check of whether the resulting model-based sentiment scores align with manually assigned labels.


In [ ]:
# Cell — Load evaluation data

DATA_PATH = Path("eval_sentences.csv")  # Edit this path if your CSV is stored elsewhere.

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. "
        "For the public repository, this file is expected to be supplied locally "
        "because it may contain copyrighted text."
    )

df = pd.read_csv(DATA_PATH, sep=";", engine="python")

# Required columns:
# - sentence_text: the sentence to evaluate
# - gold_label: manually assigned sentiment label ("positive" or "negative")
# - split: sentence role ("anchor" or "test")
required_cols = {"sentence_text", "gold_label", "split"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Present columns: {list(df.columns)}"
    )

# Standardise labels and split names.
df["gold_label"] = df["gold_label"].astype(str).str.strip().str.lower()
df["split"] = df["split"].astype(str).str.strip().str.lower()

# Keep only the binary sentiment labels used in this notebook.
df = df[df["gold_label"].isin(["positive", "negative"])].copy()

# Check that the expected split values are present.
expected_splits = {"anchor", "test"}
unexpected_splits = set(df["split"].unique()) - expected_splits

if unexpected_splits:
    raise ValueError(
        f"Unexpected split values: {unexpected_splits}. "
        "Expected only 'anchor' and 'test'."
    )

# Separate the two roles used in the notebook.
anchors = df[df["split"] == "anchor"].copy()
test = df[df["split"] == "test"].copy()

if anchors.empty:
    raise ValueError("No anchor rows found. Check that the `split` column contains 'anchor'.")

if test.empty:
    raise ValueError("No test rows found. Check that the `split` column contains 'test'.")

print("\nRows used by role:")
print(f"Anchors: {len(anchors)}")
print(f"Test:    {len(test)}")

print("\nGold-label counts:")
print(df["gold_label"].value_counts())

display(df.head(10))

> In the original experiment, anchor sentences were short manually selected positive and negative examples from the target corpus. Held-out test sentences were selected separately and used only for evaluation.

## Choose models

This notebook compares a general-purpose sentence-transformer baseline with a Tolkien-adapted sentence-transformer model.

The adapted model path is expected to point to a local model directory. The model itself is not included in this repository. Users can either update the path to their own local model or replace it with another SentenceTransformer-compatible model.

In [ ]:
# Cell — Choose models

from sentence_transformers import SentenceTransformer
from pathlib import Path

MODEL_CONFIGS = {
    "MiniLM baseline": {
        "path": "sentence-transformers/all-MiniLM-L6-v2",
        "type": "huggingface",
    },
    "Tolkien-adapted model": {
        "path": Path("../models/tolkien_sentence_transformer_epoch_1"),
        "type": "local",
    },
}

def load_model(model_name, config):
    """
    Load a SentenceTransformer model from Hugging Face or from a local path.
    """
    model_path = config["path"]

    if config["type"] == "local" and not Path(model_path).exists():
        raise FileNotFoundError(
            f"Local model path not found for '{model_name}': {model_path}\n"
            "Update MODEL_CONFIGS with the correct local path before running this notebook."
        )

    print(f"Loading model: {model_name}")
    print(f"Path: {model_path}")

    return SentenceTransformer(str(model_path))

base_model = load_model(
    "MiniLM baseline",
    MODEL_CONFIGS["MiniLM baseline"],
)

tolkien_model = load_model(
    "Tolkien-adapted model",
    MODEL_CONFIGS["Tolkien-adapted model"],
)

In [ ]:
# Cell — Helper functions

def compute_centroid(vectors: np.ndarray) -> np.ndarray:
    """
    Compute a normalized centroid from a set of sentence embeddings.

    The input embeddings are expected to already be normalized.
    The centroid is normalized again so that cosine similarity can be
    computed with a dot product.
    """
    centroid = vectors.mean(axis=0, keepdims=True)
    centroid = centroid / np.maximum(
        np.linalg.norm(centroid, axis=1, keepdims=True),
        1e-12,
    )
    return centroid


def embed_sentences(
    model: SentenceTransformer,
    sentences: list[str],
    batch_size: int = 64,
) -> np.ndarray:
    """
    Encode a list of sentences using a SentenceTransformer model.

    Embeddings are normalized so that cosine similarity is equivalent
    to the dot product.
    """
    embeddings = model.encode(
        sentences,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings


def predict_with_anchors(
    test_vectors: np.ndarray,
    positive_centroid: np.ndarray,
    negative_centroid: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Assign each test sentence to the closer sentiment anchor centroid.

    This is a nearest-centroid comparison using positive and negative
    reference sentences. It is not the same as the semantic-axis vector
    projection method used in the separate semantic-axis notebook.
    """
    similarity_positive = (test_vectors @ positive_centroid.T).reshape(-1)
    similarity_negative = (test_vectors @ negative_centroid.T).reshape(-1)

    predictions = np.where(
        similarity_positive >= similarity_negative,
        "positive",
        "negative",
    )

    return predictions, similarity_positive, similarity_negative

In [ ]:
# Cell — Evaluate one model

def evaluate_model(
    model: SentenceTransformer,
    anchors: pd.DataFrame,
    test: pd.DataFrame,
    batch_size: int = 64,
) -> dict:
    """
    Evaluate one SentenceTransformer model using positive and negative anchor sentences.

    The model embeds the anchor sentences and the held-out test sentences. Each test
    sentence is assigned to the label of the closest anchor centroid: positive or negative.
    """
    # Select anchor sentences by label.
    positive_anchor_sentences = anchors.loc[
        anchors["gold_label"] == "positive",
        "sentence_text",
    ].tolist()

    negative_anchor_sentences = anchors.loc[
        anchors["gold_label"] == "negative",
        "sentence_text",
    ].tolist()

    if not positive_anchor_sentences:
        raise ValueError("Anchors must include at least one positive sentence.")

    if not negative_anchor_sentences:
        raise ValueError("Anchors must include at least one negative sentence.")

    # Embed anchor sentences.
    positive_anchor_vectors = embed_sentences(
        model,
        positive_anchor_sentences,
        batch_size=batch_size,
    )

    negative_anchor_vectors = embed_sentences(
        model,
        negative_anchor_sentences,
        batch_size=batch_size,
    )

    # Build positive and negative anchor centroids.
    positive_centroid = compute_centroid(positive_anchor_vectors)
    negative_centroid = compute_centroid(negative_anchor_vectors)

    # Embed held-out test sentences.
    test_sentences = test["sentence_text"].tolist()
    test_vectors = embed_sentences(
        model,
        test_sentences,
        batch_size=batch_size,
    )

    # Predict labels for the held-out test sentences.
    y_true = test["gold_label"].to_numpy()
    y_pred, similarity_positive, similarity_negative = predict_with_anchors(
        test_vectors,
        positive_centroid,
        negative_centroid,
    )

    # Confusion matrix: rows = true labels, columns = predicted labels.
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["positive", "negative"],
    )

    true_positive = cm[0, 0]
    false_negative = cm[0, 1]
    false_positive = cm[1, 0]
    true_negative = cm[1, 1]

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(
            y_true,
            y_pred,
            pos_label="positive",
            zero_division=0,
        ),
        "recall_positive": recall_score(
            y_true,
            y_pred,
            pos_label="positive",
            zero_division=0,
        ),
        "f1_positive": f1_score(
            y_true,
            y_pred,
            pos_label="positive",
            zero_division=0,
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "y_true": y_true,
        "y_pred": y_pred,
        "similarity_positive": similarity_positive,
        "similarity_negative": similarity_negative,
    }

## Run evaluation for individual models

The following cells apply the same evaluation function to the MiniLM baseline and the Tolkien-adapted model. Using a shared function ensures that both models are evaluated with the same anchors, test set, and scoring procedure.

In [ ]:
# Cell — Evaluate base model

base_results = evaluate_model(
    model=base_model,
    anchors=anchors,
    test=test,
    batch_size=64,
)

base_summary = {
    key: value
    for key, value in base_results.items()
    if key not in {"y_true", "y_pred", "similarity_positive", "similarity_negative"}
}

base_summary

In [ ]:
# Cell — Confusion matrix for base model

base_cm = pd.DataFrame(
    [
        [base_results["true_positive"], base_results["false_negative"]],
        [base_results["false_positive"], base_results["true_negative"]],
    ],
    index=["true: positive", "true: negative"],
    columns=["pred: positive", "pred: negative"],
)

base_cm

In [ ]:
# Cell — Classification report for base model

base_report = classification_report(
    base_results["y_true"],
    base_results["y_pred"],
    digits=4,
    zero_division=0,
    output_dict=True,
)

base_report_df = pd.DataFrame(base_report).T
base_report_df

In [ ]:
# Cell — Show misclassified sentences for base model

pd.set_option("display.max_colwidth", None)

base_errors = test.copy()
base_errors["pred_label"] = base_results["y_pred"]
base_errors["similarity_positive"] = base_results["similarity_positive"]
base_errors["similarity_negative"] = base_results["similarity_negative"]
base_errors["margin_positive_minus_negative"] = (
    base_errors["similarity_positive"] - base_errors["similarity_negative"]
)

base_errors = base_errors[
    base_errors["pred_label"] != base_errors["gold_label"]
].copy()

base_errors = base_errors.sort_values("margin_positive_minus_negative")

base_errors[
    [
        "sentence_text",
        "gold_label",
        "pred_label",
        "similarity_positive",
        "similarity_negative",
        "margin_positive_minus_negative",
    ]
].head(30)

## Evaluate Tolkien-adapted model

This section evaluates the locally configured Tolkien-adapted model using the same anchor-based evaluation procedure as the MiniLM baseline.

In [ ]:
# Cell — Evaluate Tolkien-adapted model

tolkien_results = evaluate_model(
    model=tolkien_model,
    anchors=anchors,
    test=test,
    batch_size=64,
)

tolkien_summary = {
    key: value
    for key, value in tolkien_results.items()
    if key not in {"y_true", "y_pred", "similarity_positive", "similarity_negative"}
}

tolkien_summary

In [ ]:
# Cell — Confusion matrix for Tolkien-adapted model

tolkien_cm = pd.DataFrame(
    [
        [tolkien_results["true_positive"], tolkien_results["false_negative"]],
        [tolkien_results["false_positive"], tolkien_results["true_negative"]],
    ],
    index=["true: positive", "true: negative"],
    columns=["pred: positive", "pred: negative"],
)

tolkien_cm

In [ ]:
# Cell — Classification report for Tolkien-adapted model

tolkien_report = classification_report(
    tolkien_results["y_true"],
    tolkien_results["y_pred"],
    digits=4,
    zero_division=0,
    output_dict=True,
)

tolkien_report_df = pd.DataFrame(tolkien_report).T
tolkien_report_df

In [ ]:
# Cell — Show misclassified sentences for Tolkien-adapted model

pd.set_option("display.max_colwidth", None)

tolkien_errors = test.copy()
tolkien_errors["pred_label"] = tolkien_results["y_pred"]
tolkien_errors["similarity_positive"] = tolkien_results["similarity_positive"]
tolkien_errors["similarity_negative"] = tolkien_results["similarity_negative"]
tolkien_errors["margin_positive_minus_negative"] = (
    tolkien_errors["similarity_positive"] - tolkien_errors["similarity_negative"]
)

tolkien_errors = tolkien_errors[
    tolkien_errors["pred_label"] != tolkien_errors["gold_label"]
].copy()

tolkien_errors = tolkien_errors.sort_values("margin_positive_minus_negative")

tolkien_errors[
    [
        "sentence_text",
        "gold_label",
        "pred_label",
        "similarity_positive",
        "similarity_negative",
        "margin_positive_minus_negative",
    ]
].head(30)

## Baseline vs Tolkien-adapted model

In [ ]:
# Cell — Compare model-level results

comparison_df = pd.DataFrame(
    [
        {
            "model": "MiniLM baseline",
            "accuracy": base_results["accuracy"],
            "precision_positive": base_results["precision_positive"],
            "recall_positive": base_results["recall_positive"],
            "f1_positive": base_results["f1_positive"],
        },
        {
            "model": "Tolkien-adapted model",
            "accuracy": tolkien_results["accuracy"],
            "precision_positive": tolkien_results["precision_positive"],
            "recall_positive": tolkien_results["recall_positive"],
            "f1_positive": tolkien_results["f1_positive"],
        },
    ]
)

comparison_df

## Epoch comparison for Tolkien-adapted models

This section evaluates Tolkien-adapted models fine-tuned for different numbers of epochs. The comparison is used to identify the strongest adapted model before performing the paired comparison with the MiniLM baseline.

In [ ]:
# Cell — Evaluate Tolkien models across epochs

EPOCH_MODEL_CONFIGS = {
    "Tolkien 1 epoch": Path("../models/tolkien_sentence_transformer_epoch_1"),
    "Tolkien 2 epochs": Path("../models/tolkien_sentence_transformer_epoch_2"),
    "Tolkien 4 epochs": Path("../models/tolkien_sentence_transformer_epoch_4"),
    "Tolkien 8 epochs": Path("../models/tolkien_sentence_transformer_epoch_8"),
}

epoch_results = {}

for model_name, model_path in EPOCH_MODEL_CONFIGS.items():
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model path not found for {model_name}: {model_path}"
        )

    print(f"Loading and evaluating: {model_name}")
    model = SentenceTransformer(str(model_path))

    epoch_results[model_name] = evaluate_model(
        model=model,
        anchors=anchors,
        test=test,
        batch_size=64,
    )

In [ ]:
# Cell — Compare Tolkien models across epochs

epoch_comparison_df = pd.DataFrame(
    [
        {
            "model": model_name,
            "accuracy": results["accuracy"],
            "precision_positive": results["precision_positive"],
            "recall_positive": results["recall_positive"],
            "f1_positive": results["f1_positive"],
            "true_positive": results["true_positive"],
            "false_positive": results["false_positive"],
            "false_negative": results["false_negative"],
            "true_negative": results["true_negative"],
        }
        for model_name, results in epoch_results.items()
    ]
)

epoch_comparison_df

In [ ]:
# Cell — Plot epoch comparison

import matplotlib.pyplot as plt

# Keep only the epoch-specific Tolkien models.
plot_df = epoch_comparison_df.copy()

# Extract epoch number from labels like "Tolkien 1 epoch", "Tolkien 2 epochs", etc.
plot_df["epoch"] = (
    plot_df["model"]
    .str.extract(r"(\d+)")
    .astype(int)
)

plot_df = plot_df.sort_values("epoch")

plt.figure(figsize=(7, 4.5))

plt.plot(
    plot_df["epoch"],
    plot_df["precision_positive"],
    marker="o",
    linewidth=2,
    label="Precision",
)

plt.plot(
    plot_df["epoch"],
    plot_df["recall_positive"],
    marker="o",
    linewidth=2,
    label="Recall",
)

plt.plot(
    plot_df["epoch"],
    plot_df["f1_positive"],
    marker="o",
    linewidth=2,
    label="F1",
)

# Highlight the best F1 score.
best_idx = plot_df["f1_positive"].idxmax()
best_epoch = plot_df.loc[best_idx, "epoch"]
best_f1 = plot_df.loc[best_idx, "f1_positive"]

plt.scatter(
    best_epoch,
    best_f1,
    s=90,
    zorder=5,
)

plt.annotate(
    f"Best F1: {best_f1:.3f}",
    xy=(best_epoch, best_f1),
    xytext=(best_epoch + 0.2, best_f1 + 0.01),
    arrowprops={"arrowstyle": "->"},
)

plt.xticks(plot_df["epoch"])
plt.xlabel("Fine-tuning epoch")
plt.ylabel("Score")
plt.title("Tolkien-adapted model performance across epochs")
plt.ylim(0.70, 0.90)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
# Select the Tolkien model used for final comparison.
SELECTED_MODEL_NAME = "Tolkien 1 epoch"

selected_tolkien_results = epoch_results[SELECTED_MODEL_NAME]
selected_tolkien_path = EPOCH_MODEL_CONFIGS[SELECTED_MODEL_NAME]

selected_tolkien_model = SentenceTransformer(str(selected_tolkien_path))

print(f"Selected model for final comparison: {SELECTED_MODEL_NAME}")

In [ ]:
# Cell — Compare baseline and selected Tolkien model

baseline_vs_selected_df = pd.DataFrame(
    [
        {
            "model": "MiniLM baseline",
            "accuracy": base_results["accuracy"],
            "precision_positive": base_results["precision_positive"],
            "recall_positive": base_results["recall_positive"],
            "f1_positive": base_results["f1_positive"],
        },
        {
            "model": "Tolkien-adapted model (1 epoch)",
            "accuracy": selected_tolkien_results["accuracy"],
            "precision_positive": selected_tolkien_results["precision_positive"],
            "recall_positive": selected_tolkien_results["recall_positive"],
            "f1_positive": selected_tolkien_results["f1_positive"],
        },
    ]
)

#selected_tolkien_results = the evaluation numbers/predictions for the selected model.

baseline_vs_selected_df

## Paired model comparison

McNemar's test is used here because both models are evaluated on the same held-out test sentences. The test compares whether the two models make different errors on the same examples, rather than only comparing aggregate accuracy scores.

After comparing the epoch-specific Tolkien models, the selected adapted model is compared with the MiniLM baseline.

In [ ]:
# Cell — McNemar's test: MiniLM baseline vs selected Tolkien-adapted model

from statsmodels.stats.contingency_tables import mcnemar

y_true_arr = np.asarray(base_results["y_true"])
base_pred_arr = np.asarray(base_results["y_pred"])
selected_pred_arr = np.asarray(selected_tolkien_results["y_pred"])

if not np.array_equal(y_true_arr, np.asarray(selected_tolkien_results["y_true"])):
    raise ValueError("The two models do not appear to have been evaluated on the same test set.")

base_correct = base_pred_arr == y_true_arr
selected_correct = selected_pred_arr == y_true_arr

a = np.sum(base_correct & selected_correct)
b = np.sum(base_correct & ~selected_correct)
c = np.sum(~base_correct & selected_correct)
d = np.sum(~base_correct & ~selected_correct)

mcnemar_table = pd.DataFrame(
    [[a, b], [c, d]],
    index=["MiniLM correct", "MiniLM wrong"],
    columns=["Selected Tolkien correct", "Selected Tolkien wrong"],
)

display(mcnemar_table)

mcnemar_result = mcnemar(mcnemar_table.to_numpy(), exact=True)

print(f"McNemar p-value: {mcnemar_result.pvalue:.4f}")

if mcnemar_result.pvalue < 0.05:
    print("Result: The difference between the two models is statistically significant.")
else:
    print("Result: The difference between the two models is not statistically significant.")

## Qualitative retrieval comparison

This section compares nearest-neighbour retrieval results between the MiniLM baseline model and the selected Tolkien-adapted model.

The retrieval test is qualitative. It is used to inspect whether the adapted model retrieves sentences that are semantically closer to the query within the literary domain. It is not used as a classification metric.

The full corpus text is not included in this repository because it contains copyrighted material. To run this section locally, provide a sentence-per-line corpus file and update `CORPUS_PATH`.

In [ ]:
# Cell — Retrieval helper functions

from pathlib import Path
import numpy as np
import pandas as pd

CORPUS_PATH = Path("LotR.txt")  # Replace with your local sentence-per-line corpus file.

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CORPUS_PATH}. "
        "This file is not included in the public repository because it contains copyrighted text."
    )

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    corpus_sentences = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(corpus_sentences)} corpus sentences.")


def encode_corpus(
    model: SentenceTransformer,
    sentences: list[str],
    batch_size: int = 64,
) -> np.ndarray:
    """
    Encode a sentence-level corpus with normalized embeddings.
    """
    return model.encode(
        sentences,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def retrieve_similar_sentences(
    model: SentenceTransformer,
    query: str,
    corpus_sentences: list[str],
    corpus_embeddings: np.ndarray,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    Retrieve the top-k most similar corpus sentences for a query.

    Embeddings are normalized, so cosine similarity is computed as a dot product.
    """
    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    similarities = corpus_embeddings @ query_embedding.reshape(-1)

    top_indices = np.argsort(similarities)[::-1][:top_k]

    return pd.DataFrame(
        {
            "rank": range(1, top_k + 1),
            "sentence": [corpus_sentences[i] for i in top_indices],
            "similarity": [similarities[i] for i in top_indices],
        }
    )

In [ ]:
# Cell — Retrieval with MiniLM baseline

query = "As they came out again into the open country a great wind came."

base_corpus_embeddings = encode_corpus(
    model=base_model,
    sentences=corpus_sentences,
    batch_size=64,
)

base_retrieval_results = retrieve_similar_sentences(
    model=base_model,
    query=query,
    corpus_sentences=corpus_sentences,
    corpus_embeddings=base_corpus_embeddings,
    top_k=10,
)

pd.set_option("display.max_colwidth", None)
base_retrieval_results

In [ ]:
# Cell — Retrieval with selected Tolkien-adapted model

tolkien_corpus_embeddings = encode_corpus(
    model=selected_tolkien_model,
    sentences=corpus_sentences,
    batch_size=64,
)

tolkien_retrieval_results = retrieve_similar_sentences(
    model=selected_tolkien_model,
    query=query,
    corpus_sentences=corpus_sentences,
    corpus_embeddings=tolkien_corpus_embeddings,
    top_k=10,
)

pd.set_option("display.max_colwidth", None)
tolkien_retrieval_results

In [ ]:
# Cell — Compare retrieval rankings

retrieval_comparison = pd.DataFrame(
    {
        "rank": base_retrieval_results["rank"],
        "MiniLM sentence": base_retrieval_results["sentence"],
        "MiniLM similarity": base_retrieval_results["similarity"],
        "Tolkien-adapted sentence": tolkien_retrieval_results["sentence"],
        "Tolkien-adapted similarity": tolkien_retrieval_results["similarity"],
    }
)

retrieval_comparison